# 🎥 Wan2.1 T2V 1.3B — Google Colab Test

Free/low-cost alternative to Kaggle for the first real text-to-video test. This uses the official Wan2.1 Diffusers pipeline, not image animation.

**Target:** real human walking + visible feet/arms + physical camera movement + cinematic reveal, 480P vertical.

In [ ]:
!nvidia-smi
!pip -q install -U 'diffusers>=0.35.0' transformers accelerate ftfy imageio imageio-ffmpeg


In [ ]:
import torch, numpy as np
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID='Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae=AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe=WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
pipe.scheduler=UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
print('Wan2.1 pipeline ready')


In [ ]:
PROMPT='''Photorealistic live-action cinematic video, vertical composition. A real young man wearing a dark jacket and jeans walks naturally down an abandoned underground railway platform. His feet visibly take continuous alternating steps, knees bend naturally, arms swing with every step, shoulders and body weight shift realistically, and his jacket moves with walking. The same man remains consistent throughout. The camera starts several meters behind and slightly above him, then physically descends and smoothly tracks forward at his walking speed. The camera passes close to concrete pillars and foreground objects with strong real parallax. He notices a faint warm light, slows down, approaches a dark maintenance doorway, opens it, and reveals a gigantic hidden underground city far below filled with distant lights and enormous structures. He stops naturally in surprise while the camera continues forward past him toward the reveal. Continuous real human motion, realistic feet and hands, believable physics, natural cloth motion, realistic shadows, cinematic depth, natural motion blur, premium live-action photography, no slideshow, no static image, no digital zoom, no frozen body, no CGI-looking human, no cartoon, no text, no logo, no watermark.'''
NEGATIVE='''static, still image, slideshow, frozen body, deformed hands, extra fingers, extra legs, three legs, distorted face, bad anatomy, walking backwards, jitter, camera shake, CGI, cartoon, painting, text, logo, watermark'''
print(PROMPT)


In [ ]:
generator=torch.Generator('cuda').manual_seed(12345)
output=pipe(prompt=PROMPT, negative_prompt=NEGATIVE, height=480, width=832, num_frames=81, num_inference_steps=30, guidance_scale=5.0, generator=generator).frames[0]
export_to_video(output,'wan_colab_test.mp4',fps=16)
print('DONE: wan_colab_test.mp4')


In [ ]:
from IPython.display import Video, display
display(Video('wan_colab_test.mp4', embed=True, width=360))
